# Practice 3 — RAG & Vector Search

Practise embeddings, similarity, FAISS, and retrieval from documents.

**Topics:**
- Sentence embeddings
- Cosine similarity
- FAISS index
- Document loaders
- Chunking
- Retrieval with metadata

In [ ]:
from dotenv import load_dotenv
import os
import warnings
warnings.filterwarnings("ignore")

load_dotenv(override=True)
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")

from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("Embedding model loaded.")

## 1 — Embeddings & Cosine Similarity

See how similar sentences get similar vectors.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "A dog is playing in the park.",
    "A puppy runs in the garden.",
    "The stock market crashed today.",
    "Investors lost money in the market."
]

vectors = embeddings.embed_documents(sentences)
vectors = np.array(vectors)

sim_matrix = cosine_similarity(vectors)

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        print(f"{sim_matrix[i][j]:.3f}  |  {sentences[i]}  <->  {sentences[j]}")

## 2 — FAISS Vector Store

Store sentence vectors and search them.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

docs = [
    Document(page_content="Cats are independent pets that sleep a lot."),
    Document(page_content="Dogs are loyal and need daily walks."),
    Document(page_content="Fish are quiet pets that live in tanks."),
    Document(page_content="Birds can sing and fly around the room.")
]

vector_store = FAISS.from_documents(docs, embeddings)

results = vector_store.similarity_search("Which pet likes exercise?", k=2)
for r in results:
    print(r.page_content)

## 3 — Chunking Documents

Long documents must be split before embedding.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

long_text = '''Artificial intelligence is transforming industries.
Machine learning models learn patterns from data.
Deep learning uses neural networks with many layers.
Natural language processing helps machines understand text.
Computer vision lets machines interpret images and video.
Reinforcement learning trains agents through rewards.'''

splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=20)
chunks = splitter.create_documents([long_text])

print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks, 1):
    print(f"\nChunk {i}: {chunk.page_content}")

## 4 — RAG on a Local File

Load a text file, chunk it, store it, and ask questions.

In [ ]:
from langchain_community.document_loaders import TextLoader

# Try one of your study notes
file_path = "../study_notes/10_rag_fundamentals.md"

loader = TextLoader(file_path, encoding="utf-8")
raw_docs = loader.load()
print(f"Loaded {len(raw_docs)} document(s)")

chunks = splitter.split_documents(raw_docs)
print(f"Created {len(chunks)} chunks")

file_store = FAISS.from_documents(chunks, embeddings)

query = "What are the steps in a RAG pipeline?"
retrieved = file_store.similarity_search(query, k=3)
print(f"\nTop results for: {query}\n")
for i, chunk in enumerate(retrieved, 1):
    print(f"--- Result {i} ---")
    print(chunk.page_content[:400])
    print()

## 5 — Save and Load FAISS

Persist your vector store to disk.

In [ ]:
import shutil

index_path = "./faiss_practice_index"
shutil.rmtree(index_path, ignore_errors=True)

file_store.save_local(index_path)
print(f"Saved index to {index_path}")

loaded_store = FAISS.load_local(
    index_path,
    embeddings,
    allow_dangerous_deserialization=True
)
print("Loaded index back.")

print(loaded_store.similarity_search("RAG steps", k=1)[0].page_content[:200])

## Exercises

1. Try different chunk sizes (200, 500, 1000) and compare retrieval quality.
2. Add metadata to documents and filter retrieval by metadata.
3. Use `similarity_search_with_score` and inspect the distance scores.
4. Load a PDF instead of a text file using `PyPDFLoader`.
5. Build a simple RAG chain: `retriever | prompt | llm`.